# Drone detector training on Colab

1. Locally: `python detection/build_dataset.py --zip` and upload `data/drone/yolo.zip` to Google Drive as `MyDrive/sky-scanner/yolo.zip`.
2. Here: *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
3. Weights are saved to `MyDrive/sky-scanner/runs/<RUN>/weights/best.pt` (on Drive, so a disconnect doesn't lose them). Download into `data/drone/runs/<RUN>/weights/`.

Training settings mirror `detection/train.py`; keep them in sync.

In [ ]:
RUN = "full-n640"   # run name, as in train.py --name
MODEL = "yolo11n.pt"
IMGSZ = 640
EPOCHS = 100
DRIVE = "/content/drive/MyDrive/sky-scanner"

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount("/content/drive")
!pip install -q ultralytics==8.4.183
!rm -rf /content/yolo && unzip -q {DRIVE}/yolo.zip -d /content && ls /content/yolo

In [ ]:
from ultralytics import YOLO

YOLO(MODEL).train(
    data="/content/yolo/data.yaml",
    imgsz=IMGSZ,
    epochs=EPOCHS,
    patience=30,  # stop if val mAP hasn't improved for 30 epochs
    batch=16,
    device=0,
    hsv_h=0, hsv_s=0,  # no colour in mono images
    project=f"{DRIVE}/runs",
    name=RUN,
    exist_ok=True,
    workers=2,
)